# 49 -- flip-equivariance probe, step 1: audit the transformable rows on rung 42's own held-out set

Not a new training rung. Zero-GPU, read-only diagnostic, the same kind rung 24 ran before it
built anything. **Question:** does the currently-shipped checkpoint (rung 42 ep4,
`checkpoint-4848`, submission 03) correctly track object LOCATION when the image is
horizontally flipped at inference time -- i.e. does its answer to a quadrant/position question
mirror the way the pixels did, or does it repeat what it would have said regardless?

**Why this notebook first, no inference yet.** Rung 42 trained on a merged corpus that
promotes 30 of the public test set's 38 videos into training (`experiments/42-merged-corpus/`).
Testing flip-equivariance on those 30 videos would partly measure memorisation, not
generalisation. The only leak-free eval set for THIS checkpoint is its own declared held-out
set: **8 videos / 1,283 questions** (`experiments/42-merged-corpus/RESULTS_split_42.json`).
This notebook reuses `experiments/24-geometric-aug/_models/flip_audit.py` unchanged --
same classifier, same three camera-relative templates, same conservative
transformable/invariant/excluded_situs/manual_review dispositions -- and restricts it to those
8 videos, to get the real transformable-row count before any GPU is spent.

**Data provenance check, done before this notebook (not re-derived here):** the local FRAME
parquets used below were verified against rung 08's data card (6,252 test / 13,748 train / 38
test videos / 92 train videos / 4,486 test frames -- all match) and against
`RESULTS_split_42.json`'s own declared question count for the 8 held-out videos (1,283 -- also
matches exactly). Re-asserted in cell 4 below so a future run on different data raises instead
of silently drifting.

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, sys
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "49-flip-equivariance":
    EXP = REPO / "experiments" / "49-flip-equivariance"

# flip_audit.py is owned by experiment 24 -- reused here unchanged, not copied.
FLIP_AUDIT_SRC = REPO / "experiments" / "24-geometric-aug" / "_models"
assert FLIP_AUDIT_SRC.is_dir(), f"expected rung 24's _models at {FLIP_AUDIT_SRC}"
if str(FLIP_AUDIT_SRC) not in sys.path:
    sys.path.insert(0, str(FLIP_AUDIT_SRC))

import flip_audit
print("repo:", REPO)
print("flip_audit module:", flip_audit.__file__)

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects BELOW this cell) --------
# Candidate DATA_ROOTs, checked in order -- first one whose heico/test.parquet exists wins.
# "/workspace/orena-data": the pod's nested layout (<root>/<dataset>/{split}.parquet).
# "external_data/orena-data": the repo-relative fallback other notebooks use.
# The third is this machine's local copy (verified against rung 08's data card in cell 4
# below before anything downstream trusts it) -- flat layout, one level up from the dataset dirs.
DATA_ROOT_CANDIDATES = [
    "/workspace/orena-data",
    "external_data/orena-data",
    "/Users/yingyuyang/Desktop/MyCodes/Frame",
]

SPLIT_42_JSON = "experiments/42-merged-corpus/RESULTS_split_42.json"

In [ ]:
# --- derived: resolve DATA_ROOT, load raw parquets, load rung 42's held-out list ------------
DATA_ROOT = next(
    (d for d in (Path(c) if Path(c).is_absolute() else REPO / c for c in DATA_ROOT_CANDIDATES)
     if (d / "heico" / "test.parquet").exists()),
    None,
)
assert DATA_ROOT is not None, f"no heico/test.parquet found in any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT:", DATA_ROOT)

def _load(split: str) -> pd.DataFrame:
    parts = []
    for ds in ("heico", "lapchole"):
        df = pd.read_parquet(DATA_ROOT / ds / f"{split}.parquet")
        df["_dataset"] = ds
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

train_raw = _load("train")
test_raw = _load("test")

split_42_path = REPO / SPLIT_42_JSON
assert split_42_path.exists(), f"missing {split_42_path} -- has rung 42's split been committed?"
split_42 = json.loads(split_42_path.read_text())
held_out = split_42["videos_held_out"]
print(f"rung 42 held-out videos: {len(held_out)}")
for v in held_out:
    print(" ", v)

In [ ]:
# --- 1. shape check vs rung 08's data card -- a failing assert here is a finding, never move the expected value
EXPECTED = {
    "test rows": (len(test_raw), 6252),
    "train rows": (len(train_raw), 13748),
    "test videos": (test_raw["video"].nunique(), 38),
    "train videos": (train_raw["video"].nunique(), 92),
    "test frames": (test_raw.groupby(["video", "timestamp_start"]).ngroups, 4486),
}
for name, (got, want) in EXPECTED.items():
    print(f"  {name:14s} got={got:<8} want={want:<8} {'OK' if got == want else 'MISMATCH'}")
    assert got == want, f"DATA MISMATCH: {name} = {got}, rung 08's data card says {want}"
print("\nshape matches rung 08's data card -- this is the canonical FRAME test/train split.")

In [ ]:
# --- 2. the 8 rung-42 held-out videos: present, and the count matches the submission's own number
test_raw["_video_key"] = test_raw["_dataset"] + "/" + test_raw["video"]
held_out_mask = test_raw["_video_key"].isin(held_out)
held_out_test = test_raw[held_out_mask].copy()

for v in held_out:
    n = (test_raw["_video_key"] == v).sum()
    assert n > 0, f"held-out video not found in local test.parquet: {v}"
    print(f"  FOUND n_questions={n:4d}  {v}")

print(f"\ntotal questions across the 8 held-out videos: {len(held_out_test)}")
assert len(held_out_test) == 1283, (
    f"expected 1,283 questions (submission 03's own number), got {len(held_out_test)}"
)
print("matches submission 03's declared 1,283 -- this is rung 42's own held-out set.")

In [ ]:
# --- 3. run flip_audit, restricted to the 8 held-out videos -------------------------------
# Audited per-dataset, passing the real `dataset=` value -- feeding a pre-concatenated frame
# without it silently stamps every row dataset=None (the bug 24_position_prior_probe.ipynb's
# own _audit_split comment documents). Same defensive pattern reused here.
parts = []
for ds in ("heico", "lapchole"):
    sub = held_out_test[held_out_test["_dataset"] == ds]
    if len(sub):
        parts.append(flip_audit.audit_dataframe(sub, split="test", dataset=ds))
audit = pd.concat(parts, ignore_index=True)

assert len(audit) == len(held_out_test), "audit row count doesn't match the held-out subset"

summary = (
    audit.groupby(["dataset", "disposition", "rule"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["disposition", "rule", "dataset"])
)
print(summary.to_string(index=False))
print()
print(audit["disposition"].value_counts())

In [ ]:
# --- 4. persist -- row-level audit + disposition summary, at the experiment root ----------
rows_path = EXP / "RESULTS_flip_audit_heldout8.csv"
summary_path = EXP / "RESULTS_flip_audit_heldout8_summary.csv"
audit.to_csv(rows_path, index=False)
summary.to_csv(summary_path, index=False)
print("wrote", rows_path)
print("wrote", summary_path)

n_transformable = int((audit["disposition"] == "transformable").sum())
print(f"\ntransformable rows on rung 42's 8 held-out videos: {n_transformable}")
print(f"(for reference: rung 24's original audit found 871 transformable rows on all 38 test videos)")